# Case 300 Powerflow Solver Test

FIXME: skipped, dpsim-simulator/cim-grid-data's case300.xml has bad data (transformer TN143_T7PwerTrafo_143-148 has the same rated voltage on both windings). Re-enable once a corrected source is available.

## Run simulation

In [ ]:
import requests

url = "https://raw.githubusercontent.com/dpsim-simulator/cim-grid-data/master/Matpower_cases/case300.xml"
filename = "case300.xml"
with open(filename, "wb") as out_file:
    content = requests.get(url, stream=True).content
    out_file.write(content)

files = [filename]

In [ ]:
from villas.dataprocessing.readtools import *
import dpsimpy

In [ ]:
name = "case300"
reader = dpsimpy.CIMReader(name)
system = reader.loadCIM(
    50, files, dpsimpy.Domain.SP, dpsimpy.PhaseType.Single, dpsimpy.GeneratorType.PVNode
)
system

In [ ]:
sim = dpsimpy.Simulation(name, dpsimpy.LogLevel.info)
sim.set_system(system)
sim.set_domain(dpsimpy.Domain.SP)
sim.set_solver(dpsimpy.Solver.NRP)
sim.set_pf_keep_last_solution(False)
sim.set_time_step(1.0)
sim.set_final_time(10.0)

logger = dpsimpy.Logger(name)
sim.add_logger(logger)
for node in system.nodes:
    logger.log_attribute(node.name() + ".V", "v", node);

In [ ]:
sim.run()

## Read DPsim results

In [ ]:
dpsim_result_file = "logs/" + name + ".csv"
ts_dpsim = read_timeseries_csv(dpsim_result_file)

In [ ]:
for k, v in ts_dpsim.items():
    print(v.name + ":" + str(v.values[0]))